In [1]:
#!/usr/bin/env python
# coding: utf-8
import sys
sys.path.append("../")
sys.path.append("../Visualization/")
sys.path.append("../../")

sys.path.append("../../experiments/parametrization_experiments/")

sys.path.append('../../'); sys.path.append('../../../'); sys.path.append('../../../gmsh'); sys.path.append('../../Visualization/')


sys.path.append('periodic_patches/')
sys.path.append('gmsh')
import MeshFEM, mesh, sparse_matrices, benchmark, field_sampler, mesh_utilities
import inflatables_parametrization as parametrization, numpy as np, importlib, pickle, wall_generation
import utils
import py_newton_optimizer
from py_newton_optimizer import NewtonOptimizerOptions
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization, wall_width_formulas as wwf
from tri_mesh_viewer import TriMeshViewer

import parallelism
parallelism.set_max_num_tbb_threads(4)
import parametrization_experiment_helper
import os 
import inflation
from mesh_utilities import SurfaceSampler, tubeRemesh
import boundaries
import time 

data_time_stamp = '2024_01_18_00_03'

output_time_stamp = time.strftime("%Y_%m_%d_%H_%M")


   

In [4]:
shape_index = 1
pattern_index = 1


experiment_file, stiffness_path, pattern_name, num_pattern_params, param_index, default_param, param_range, param_normalization_factor, fusing_curve_polyline, shape_name, shape_path, use_holes = parametrization_experiment_helper.parse_input(shape_index, pattern_index)

print("Running experiment for shape {} and pattern {}".format(shape_name, pattern_name))

meshing_data_path = 'output/{}/{}_{}/'.format(data_time_stamp, shape_name, pattern_name)
sdf_data_path = '../../experiments/parametrization_experiments/output/{}/{}_{}/'.format(data_time_stamp, shape_name, pattern_name)
optimization_data_path = 'output/optimization/{}/{}_{}/'.format(output_time_stamp, shape_name, pattern_name)
if not os.path.exists(optimization_data_path):
    os.makedirs(optimization_data_path)

target_surf = mesh.Mesh(shape_path)
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=False))
target_surf = mesh_utilities.subdivide_loop(target_surf, 1)

m = MeshFEM.mesh.Mesh(meshing_data_path + '/parametrized_mesh.obj')
vertices = m.vertices()
m = MeshFEM.mesh.Mesh(np.concatenate((vertices, np.zeros((len(vertices), 1))), axis = 1), m.elements())
fusing_data = np.load(meshing_data_path + '/fusing_data.npy')

isheet = inflation.InflatableSheet(m, fusing_data)
isheet.setUseTensionFieldEnergy(True)
isheet.setUseHessianProjectedEnergy(False)
isheet.pressure = 0.025
bdryVars = boundaries.getOuterBoundaryVars(isheet)

fixedVars_list = [bdryVars, []]
tag_name = ['fixed_boundary', 'free_boundary']

uv = np.load(sdf_data_path + '/rparam_uv.npy')
paramSampler = SurfaceSampler(np.pad(uv, [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), target_surf.vertices())

opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.niter = 2000
opts.gradTol = 1e-7

hessianShift = 1e-6

viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)
viewer.setCameraParams(((1.613494603240345, -3.9332708615926393, 1.4922998234349831),
(-0.05948468564942635, 0.33267929672385665, 0.941162078339598),
(0.0, 0.0, 0.0)))
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

In [5]:
viewer.show()

In [6]:
from visualization import TriMeshViewerWithSurface

In [7]:
framerate = 10
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(isheet)[:, 0])


fixedVars = fixedVars_list[0]
tag = tag_name[0]
# isheet.setVars(np.load(meshing_data_path + '/{}_inflated_sheet_vars.npy'.format(tag)))
# Reset the inflation and set up target-attraction forces
isheet.setUninflatedDeformation(liftedSheetPositions.transpose(), prepareRigidMotionPinConstraints=False)
targetAttractedSheet = inflation.TargetAttractedInflation(isheet, target_surf)
targetAttractedSheet.energy(targetAttractedSheet.EnergyType.Fitting)

targetAttractedSheet.targetSurfaceFitter().holdClosestPointsFixed = True
targetAttractedSheet.fittingWeight = 1e-5

# Re-inflate, this time applying target-attraction forces.
benchmark.reset()
cr = inflation.inflation_newton(targetAttractedSheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)
benchmark.report()

print("setup sheet")
import sheet_optimizer, opt_config
origDesignMesh = isheet.mesh().copy()

def config(so):
    fcs = so.rso.fusingCurveSmoothness()
    fcs.interiorWeight = 1/10


In [8]:

sheet_opt = sheet_optimizer.PySheetOptimizer(targetAttractedSheet, fixedVars, renderMode=sheet_optimizer.RenderMode.OFFSCREEN, screenshotPath=optimization_data_path + '{}_{}_{}.mp4'.format(tag, shape_name, pattern_name),
                                            detActivationThreshold=0.9, detActivationThresholdTubeTri=0.5,
                                            originalDesignMesh=origDesignMesh, fusingCurveSmoothnessConfig=opt_config.FusingCurveSmoothnessParams(0.0, 0.0, 1.0, 1.0), customConfigCallback=config)

In [9]:
sheet_opt.rso.getEquilibriumSolver().options.niter = 20


In [12]:
nds = sheet_opt.targetAttractedSheet.nondimensionalization

In [19]:
nds.smoothingScale(), nds.fittingEnergyScale(), nds.wallWidth, nds.fusedArea

In [21]:
sheet = sheet_opt.targetAttractedSheet.sheet()

In [23]:
len(sheet.deformedWallVertexPositions())

In [43]:
tas = sheet_opt.targetAttractedSheet

In [44]:
tas.sheet()

In [45]:
sheet_opt.rso.getEquilibriumSolver().options.niter = 1

In [46]:
tas = sheet_opt.tar

In [47]:
tas

In [48]:
sheet_opt.optimize()a



In [16]:
sheet_opt.setSolver(sheet_optimizer.Solver.SCIPY, 2)
sheet_opt.deploy_viewer.setCameraParams(((2.2283129063050544, -2.4424908418018645, 1.480176156096905),
    (-0.20481495134223526, 0.5059968464853937, 0.8378651604247005),
    (-0.10862883029189645, -0.1146986958685825, -0.4968676111207299)))
sheet_opt.flat_viewer.setCameraParams(((-0.005724654779466134, -0.15742800643532207, 3.663779058858404),
    (0.0, 1.0, 0.0),
    (-0.005724654779466134, -0.15742800643532207, 0.0)))
sheet_opt.flat_viewer.showWireframe()

sheet_opt.deploy_viewer.scalarFieldGetter = visualization.ISheetScalarField.TGT_DIST(targetAttractedSheet.sheet(), target_surf)

sheet_opt.optimize()
sheet_opt.save(optimization_data_path + '{}_{}_{}.pkl.gz'.format(tag, shape_name, pattern_name))

In [4]:
run_experiment(0, 1)